# Training and evaluating DistilBERT on headline level classification

In [1]:
import json
import pandas as pd
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

data_path = "../data"
models_path = "../models/DistilBERT_headline_runs"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

f:\Mike\Documents\Backup\Python\fake-headline-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1008 13:30:37.194000 2160 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


### Transform data into appropriate format

Create Python list of dicts

In [2]:
def flatten_transform(target_set):

    modified_set = []

    for item in target_set:
        date = item["date"]
        answer = item["answer"]
        for index, headline in enumerate(item["headlines"]):
            if index == answer:
                fake = 1
            else:
                fake = 0
            entry = {
                "Date": date,
                "Headline": headline,
                "Fake": fake
            }
            modified_set.append(entry)

    return modified_set

train_set_nn_ready = flatten_transform(train_nn_set)
validate_set_nn_ready = flatten_transform(validate_set)
test_set_nn_ready = flatten_transform(test_set)

Transform into dataframe and create X and y classes

In [3]:
train_nn_df = pd.DataFrame(train_set_nn_ready)
X_train_nn_text = train_nn_df["Headline"]
y_train = train_nn_df["Fake"]

validate_nn_df = pd.DataFrame(validate_set_nn_ready)
X_validate_nn_text = validate_nn_df["Headline"]
y_validate = validate_nn_df["Fake"]

test_df = pd.DataFrame(test_set_nn_ready)
X_test_nn_text = test_df["Headline"]
y_test = test_df["Fake"]

### DistilBERT

Initial setup and tokenisation

In [4]:
# Set the seed
seed = 42
set_seed(seed)

# Choose model
model_name = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define tokenizer settings (need to think about this)
truncation = True
padding = "max_length"
max_length = 50

# Tokenize dataset
train_encodings = tokenizer(
    list(X_train_nn_text),
    truncation=truncation,
    padding=padding,
    max_length=max_length
    )
validate_encodings = tokenizer(
    list(X_validate_nn_text),
    truncation=truncation,
    padding=padding,
    max_length=max_length
    )
test_encodings = tokenizer(
    list(X_test_nn_text),
    truncation=truncation,
    padding=padding,
    max_length=max_length
    )

# Convert to torch Dataset
class HeadlineDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Set up train, validation and test datasets
train_dataset = HeadlineDataset(train_encodings, list(y_train))
validate_dataset = HeadlineDataset(validate_encodings, list(y_validate))
test_dataset = HeadlineDataset(test_encodings, list(y_test))

# Size of train and test datasets
n_train = len(train_dataset)
n_validate = len(validate_dataset)
n_test = len(test_dataset)

In [32]:
token_lengths = [len(tokenizer.encode(text)) for text in X_validate_nn_text]
print(max(token_lengths))
print(np.percentile(token_lengths, [50, 90, 95, 99]))

47
[17.   24.   26.   32.99]


Model hyperparameters and training (only run in Colab)

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
num_model_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Define metadata saving functions
def create_run_dir(base=models_path):
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base) / timestamp
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

# Create run directories
run_dir = create_run_dir()

# Define training args
output_dir = str(run_dir / "checkpoints")
num_train_epochs=20
per_device_train_batch_size=32
per_device_eval_batch_size=268
eval_strategy="epoch"
eval_steps=1
save_strategy="epoch"
save_steps=1
disable_tqdm=True
logging_steps=1
report_to="none"
learning_rate=5e-5 # Default is 5e-5
optim="adamw_torch" # Default is "adamw_torch"
weight_decay=0 # Default is 0
warmup_steps=0 # Default is 0
lr_scheduler_type="linear" # Default is "linear"
load_best_model_at_end=True
metric_for_best_model="eval_loss"
greater_is_better=False
early_stopping_patience=5

# Assign training args
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    eval_strategy=eval_strategy,
    eval_steps=eval_steps,
    save_strategy=save_strategy,
    disable_tqdm=disable_tqdm,
    logging_steps=logging_steps,
    report_to=report_to,
    learning_rate=learning_rate,
    optim=optim,
    weight_decay=weight_decay,
    warmup_steps=warmup_steps,
    lr_scheduler_type=lr_scheduler_type,
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    greater_is_better=greater_is_better
)

# Early stopping option
callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)] # If "None", comment line in Trainer

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validate_dataset,
    callbacks=callbacks
)

# Train
output = trainer.train()

# Best validation loss and the step when it happened
best_step = trainer.state.best_step if hasattr(trainer.state, "best_step") else None
best_metric = trainer.state.best_metric if hasattr(trainer.state, "best_metric") else None

# Tiered run metadata
run_metadata = {
    "Model Information": {
        "Model name": model_name,
        "Number of model parameters": num_model_params,
        "Number of trainable parameters": trainable_params,
    },
    "Dataset Information": {
        "Training set size": n_train,
        "Validation set size": n_validate,
        "Test set size": n_test,
    },
    "Tokenizer Settings": {
        "Truncation setting": truncation,
        "Padding setting": padding,
        "Max length": max_length,
    },
    "Training Hyperparameters": {
        "Number of training epochs": num_train_epochs,
        "Per-device training batch size": per_device_train_batch_size,
        "Per-device eval batch size": per_device_eval_batch_size,
        "Number of evaluation steps": eval_steps,
        "Output directory": output_dir,
        "Save strategy": save_strategy,
        "Save steps": save_steps,
        "Disable tqdm": disable_tqdm,
        "Logging steps": logging_steps,
        "Report to": report_to,
        "Learning rate": learning_rate,
        "Optimiser": optim,
        "Weight decay": weight_decay,
        "Warmup steps": warmup_steps,
        "Learning rate scheduler type": lr_scheduler_type,
        "Callbacks": [type(cb).__name__ for cb in callbacks],
        "Seed": seed,
        "Load best model at end": load_best_model_at_end,
        "Metric for best model": metric_for_best_model,
        "Greater is better": greater_is_better,
        "Early stopping patience": early_stopping_patience,
    },
    "Training Results": {
        "Training time and speed": output.metrics,
    }
}

def convert_numpy(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (np.floating, float)):
        return round(float(obj), 6)
    else:
        return obj

with open(f"{run_dir}/run_metadata.json", "w") as f:
    json.dump(convert_numpy(run_metadata), f, indent=4)

print(f"Run metadata saved to {run_dir}/run_metadata.json")

# Extract log history
log_history = trainer.state.log_history

# Training loss
train_points = [x for x in log_history if "loss" in x]
train_steps = [p["step"] for p in train_points]
train_losses = [p["loss"] for p in train_points]

# Validation loss
val_points = [x for x in log_history if "eval_loss" in x]
val_steps = [p["step"] for p in val_points]
val_losses = [p["eval_loss"] for p in val_points]

# Plot
plt.figure(figsize=(8,5))
plt.plot(train_steps, train_losses, label="Training Loss")
plt.plot(val_steps, val_losses, label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(run_dir / "loss_curve.png")
plt.close()



Logging and storing (only run in Colab)

In [ ]:
log_history = trainer.state.log_history

clean_log = []
last_train_loss = None
last_step = None

# Record values at each logging step
for entry in log_history:
    step = entry.get("step")
    if "loss" in entry:
        last_train_loss = entry["loss"]
    if "eval_loss" in entry:
        # Skip if this step is already appended
        if step == last_step:
            continue
        clean_entry = {
            "step": step,
            "train_loss": last_train_loss,
            "eval_loss": entry.get("eval_loss"),
            "epoch": entry.get("epoch")
        }
        clean_log.append(clean_entry)
        last_step = step

# Save JSON
with open(run_dir / "log_history.json", "w") as f:
    json.dump(clean_log, f, indent=2)

# Save CSV
df = pd.DataFrame(clean_log)
df.to_csv(run_dir / "log_history.csv", index=False)

# Save best model
best_model_dir = run_dir / "best_model"
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

Load model for evaluation

In [5]:
model_path = Path(f"{models_path}/20260922-123727/best_model")
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

test_inputs = {
    key: torch.tensor(value)
    for key, value in test_encodings.items()
}

with torch.no_grad():
    outputs = model(**test_inputs)

logits = outputs.logits

Loading weights: 100%|██████████| 104/104 [00:01<00:00, 73.05it/s]


In [6]:
test_df["Fake Logit DistilBERT"] = logits[:, 1].cpu().numpy()
test_by_quiz_df = test_df.groupby("Date")

correct = 0

for date, quiz_df in test_by_quiz_df:
    predicted_index = quiz_df["Fake Logit DistilBERT"].idxmax()
    actual_index = quiz_df["Fake"].idxmax()

    if predicted_index == actual_index:
        correct += 1

accuracy_DistilBERT_headline = correct / len(test_set)

print(f"Quiz-level accuracy: {accuracy_DistilBERT_headline:.2%}")

Quiz-level accuracy: 49.25%


In [7]:
# Calculate probabilites using softmax
fake_probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

# Reshape headline-level probabilities into 67 quizzes × 4 headlines
fake_probs = fake_probs.reshape(-1, 4)

# Predicted fake headline position (0–3)
y_pred_quiz = np.argmax(fake_probs, axis=1)

# Confidence = fake probability assigned to the selected headline
confidence = np.max(fake_probs, axis=1)

# Dates from the fixed test set
test_dates = [quiz["date"] for quiz in test_set]

# Create prediction dataframe
predictions_DistilBERT = pd.DataFrame({
    "date": test_dates,
    "prediction": y_pred_quiz,
    "confidence": confidence
})

predictions_DistilBERT.to_csv(
    "../data/predictions_distilbert_headline.csv",
    index=False
)

In [10]:
predictions = {}

for date, quiz_df in test_by_quiz_df:
    predictions[date] = quiz_df["Fake Logit DistilBERT"].idxmax()

joblib.dump(predictions, f"{data_path}/DistilBERT_predictions.pkl")

['../data/DistilBERT_predictions.pkl']

In [11]:
nn_predictions = joblib.load(f"{data_path}/vnn_predictions.pkl")
DB_predictions = joblib.load(f"{data_path}/DistilBERT_predictions.pkl")

same_prediction = sum(
    nn_predictions[date] == DB_predictions[date]
    for date in nn_predictions
)

print(f"Same prediction: {same_prediction}/{len(nn_predictions)}")

Same prediction: 42/67


In [25]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
comparison_df = joblib.load(f"{data_path}/comparison_df.pkl")
comparison_df["Fake Prob DistilBERT"] = torch.sigmoid(logits[:, 1])
# comparison_df

In [ ]:
model_cols = ["Fake Prob BoW LR", "Fake Prob SE LR", "Fake Prob VNN", "Fake Prob DistilBERT"]
for col in model_cols:
    comparison_df[col] = (
        comparison_df.groupby("Date")[col]
        .transform(lambda x: (x == x.max()).astype(int))
    )
    comparison_df.rename(columns={col: col.replace("Prob", "Pred")}, inplace=True)

comparison_df